# Forecast Demanda Omnichannel - Analysis Notebook

## Objetivo
Prever demanda diaria por SKU/canal para apoiar reposicao e planejamento de estoque.

## Perguntas de negocio
- Como esta o erro por horizonte?
- Existem efeitos sazonais e de promocao?
- Quais SKUs merecem politica diferente de estoque?


### O que este código faz
Carrega bibliotecas, lê os arquivos principais do projeto (`data` e `metrics`) e mostra uma primeira amostra dos dados para conferência inicial.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path('..')
data_path = project_root / 'data' / 'demand_history_synthetic.csv'
metrics_path = project_root / 'models' / 'metrics.json'

pd.set_option('display.max_columns', 200)
df = pd.read_csv(data_path)
metrics = json.loads(metrics_path.read_text(encoding='utf-8')) if metrics_path.exists() else {}

print('shape:', df.shape)
print('metrics keys:', list(metrics.keys()))
df.head(5)


### O que este código faz
Verifica qualidade básica dos dados: valores ausentes e distribuição da variável-alvo (quando existir). É um check rápido antes de qualquer análise.


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print('missing columns:', len(missing))
if len(missing) > 0:
    display(missing)

if 'demand_units' in df.columns:
    print('target distribution:')
    print(df['demand_units'].value_counts(normalize=True).round(4))


### O que este código faz
Realiza uma análise exploratória específica do projeto para gerar visão inicial dos padrões principais.


In [ ]:
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])

base_group = 'sku' if 'sku' in df.columns else ('product_id' if 'product_id' in df.columns else None)
if base_group:
    agg = df.groupby(base_group)['demand_units'].agg(['mean', 'std', 'min', 'max']).sort_values('mean', ascending=False)
    agg.head(15)
else:
    df['demand_units'].describe()


### O que este código faz
Gera visualizações complementares para facilitar interpretação e comunicação dos resultados.


In [ ]:
if 'date' in df.columns:
    ts = df.groupby('date')['demand_units'].sum().sort_index()
    plt.figure(figsize=(11, 4))
    plt.plot(ts.index, ts.values)
    plt.title('Demanda total por dia')
    plt.grid(alpha=0.2)
    plt.tight_layout()


### O que este código faz
Exibe as métricas salvas no pipeline para conectar análise exploratória com desempenho final do modelo.


In [ ]:
pd.DataFrame([metrics]).T.rename(columns={0: 'value'}).head(30)


## Status atual
- Este projeto ja foi executado de ponta a ponta.
- Artefatos (dados, modelos, metricas e relatorios) estao gerados.
- Repositorio publicado e sincronizado no GitHub.
